
# DL Assignment 03

**Name:** Joyant Sheikhar Gupta Joy

**Course Email:**  joyantsheikharguptajoy@gmail.com


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:


### **Chosen Dataset**

For this assignment, I selected the [Pima Indians Diabetes Dataset](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

This dataset is suitable for this task because:
* It is a binary classification problem (diabetes: 0 or 1)
* It contains 768 samples, which is more than the required 300 samples.
* It includes multiple important medical features such as glucose level, blood pressure, BMI, age, etc.
* It is available in CSV format, making it easy to load and process.

### **Step 1: Load The Dataset**

In [ ]:
import pandas as pd

df = pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


First, I loaded the dataset using pandas.

### **Step 2: Separate features and target**


In [ ]:
X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values

Here,
* `X` contains input features
* `y` contains the target variable (0 = no diabetes, 1 = diabetes)

### **Step 3: Train-Test Split**

In [ ]:
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

I split the dataset into training and testing parts using an 80/20 ratio. Training data is used to train the model, while testing data is kept unseen during training to evaluate real performance

### **Step 4: Handling missing values (Training Data Only)**

In [ ]:
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols:
  col_idx = df.columns.get_loc(col)

  mean_val = X_train[:, df.columns.get_loc(col)].mean()

  X_train[:, col_idx] = [
      mean_val if x == 0 else x for x in X_train[:, col_idx]
  ]

  X_test[:, col_idx] = [
      mean_val if x == 0 else x for x in X_test[:, col_idx]
  ]

In this dataset, some features like `Glucose`,	`BloodPressure`,	`SkinThickness`, `Insulin`, `BMI` contain 0 values, which are not realistic. These are treated as missing values. So, I replaced 0 values with the mean of the non-zero values from training data only to avoid data leakage. I used the mean of non-zero values instead of the full column mean because zero values are not valid in these medical features. Replacing with mean hepls to keep the data balanced and avoids removing rows, which could reduce dataset size.

### **Step 5: Feature Scaling**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

I applied standardization (Z-score normalization) to improve model performance. Feature scaling ensures that all input features are on a similar scale. I fitted the scaler only on training data and applied the same transformation to test data to avoid data leakage.

### **Step 6: Convert to NumPy arrays**

In [ ]:
import numpy as np

X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In this step, I converted the dataset into NumPy arrays, which is a more efficient format for numerical operations and preparing data for PyTorch Tensors.

### **Step 7: Convert to PyTorch Tensors**

In [ ]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

PyTorch models require tensor format for training. So, I converted all datasets into tensors with `float32` type for compatibility with the neural network.

# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:


### **Neural Network Design**

In [ ]:
import torch.nn as nn

class DiabetesModel(nn.Module):
  def __init__(self, num_features):
    super(DiabetesModel, self).__init__()

    self.linear1 = nn.Linear(num_features, 16)  # hidden layer
    self.relu = nn.ReLU()

    self.linear2 = nn.Linear(16, 1)                      # output layer
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear1(features)
    out = self.relu(out)
    out = self.linear2(out)
    out = self.sigmoid(out)
    return out

I designed a simple neural network using PyTorch `nn.Module`. The model is built for a binary classification task, so it takes input features and predicts whether the output is 0 or 1.

### **Justification**

I used one hidden layer with 16 neurons because the dataset is not very large or complex, so a simple architecture is enough to learn the patterns without overfitting. If the model is too large, it may memorize the training data instead of learning properly.

For actication functions, I used ReLU in the hidden layer because it is simple and helps the model learn faster by reducing the vanishing gradient problem. For the output layer, I used Sigmoid because this is a binary classification problem, and sigmoid gives output between 0 and 1 which represents probability.

### **Print total trainable parametes**

In [ ]:
model = DiabetesModel(num_features=8)

total_params = 0

for p in model.parameters():
  if p.requires_grad:
    total_params += p.numel()

print("Total Trainable Parameters:", total_params)

Total Trainable Parameters: 161


# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:

### **Loss function and optimizer**

In [ ]:
import torch.optim as optim

# Loss function
loss_fn = nn.BCELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

### **Justification**

This is a classification problem, not a regression problem. In this dataset, the target value is either 0 or 1, which means the output is categorical. Because of this, regression loss functions like MSE are not suitable here. Instead, Binary Cross Entropy Loss (BCELoss) is more appropriate since it is designed for binary classfication tasks.

The nature of this dataset is also relatively simple and structured, with numerical features like glucose level, BMI, and age. Since the output is probability-based (0 to 1), label.

For the optimizer, I used Adam because it adapts the learning rate automatically and works efficiently for most deep learning models, especially when the dataset is not very large or complex.

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

### **Training loop**

In [ ]:
epochs = 100
loss_history = []

for epoch in range(epochs):
  # Shuffle training data
  indices = torch.randperm(len(X_train))
  X_train = X_train[indices]
  y_train = y_train[indices]

  # Gradient reset
  optimizer.zero_grad()

  # Forward pass
  outputs = model(X_train)

  # Loss computation
  loss = loss_fn(outputs, y_train.view(-1, 1))

  # Backward pass
  loss.backward()

  # Update parameters
  optimizer.step()

  # Store loss
  loss_history.append(loss.item())

  # Print loss every 10 epochs
  if (epoch+1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item(): .4f}")

Epoch [10/100], Loss:  0.6817
Epoch [20/100], Loss:  0.6673
Epoch [30/100], Loss:  0.6531
Epoch [40/100], Loss:  0.6389
Epoch [50/100], Loss:  0.6247
Epoch [60/100], Loss:  0.6107
Epoch [70/100], Loss:  0.5971
Epoch [80/100], Loss:  0.5838
Epoch [90/100], Loss:  0.5712
Epoch [100/100], Loss:  0.5593


I trained the model using a full training loop for 100 epochs. In each epoch, the model learns from the training data step by step.

### **Explanation of each step**

**Shuffle Training Data**

In this step, I shuffled the training data using `torch.randperm()` to change the order of samples in each epoch. This helps the model avoid learning from a fixed sequence and improves generalization.

**Gradient Reset**

Before each training step, we reset the gradients using `optimizer.zero_grad()`. This is important because PyTorch accumulates gradients by default, so without resetting, old gradients would mix with new ones.

**Forward Pass**

In the forward pass, the input training data is passed through the model. The model makes predictions based on its current weights. At this stage, we only calculate the output, not update anything.

**Loss Computation**

After getting the predictions, the loss is calculated using Binary Cross Entropy Loss (BCELoss). This tells how far the predicted values are from the actual labels.

**Backward Pass**

In this step, the model calculates gradients using `loss.backward()`. These gradients show how each parameter should change to reduce the loss.

**Parameter Update**

Finally, the optimizer (Adam) updates the model's weights using `optimizer.step()`. This step improves the model by reducing the loss based on calculated gradients.

**Loss Tracking**

The loss value is stored in a list (loss_history) so we can analyze training progess later.

**Logging**

Training loss is printed every 10 epochs to monitor how well the model is learning over time.

# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

### **Evaluation Code (Accuracy for Classification)**

In [ ]:
model.eval()

with torch.no_grad():
  outputs = model(X_test)

  # Convert probabilites to 0 or 1
  pred= (outputs >= 0.5).float()

  # Calculate accuracy
  corr = (pred == y_test.view(-1 ,1)).sum().item()
  total = y_test.size(0)

  accuracy = corr / total
  print("Test Accuracy:", accuracy)

Test Accuracy: 0.7272727272727273


**Training Accuracy (for comparison)**

In [ ]:
with torch.no_grad():
  train_out = model(X_train)
  train_pred = (train_out >= 0.5).float()

  train_corr = (train_pred == y_train.view(-1, 1)).sum().item()
  train_total = y_train.size(0)

  train_acc = train_corr / train_total
  print("Train Accuracy:", train_acc)

Train Accuracy: 0.7263843648208469


### **Traning vs Testing Performance**

During training, the model gradually learned patterns from the dataset, which is shown by the stable training process and good final performance. When I evaluated the model, both training and testing accuracies are almost the same.

The training accuracy is 0.7272, and the test accuracy is 0.7263, which are extremely close to each other. This indicates that the model is performing consistently on both seen and unseen data. It means the model has learned general patterns from the dataset rather than memorizing the training data.

### **Underfitting or Overfitting**

Based on the results, the model does not show strong signs of overfitting or underfitting. If the model was overfitting, the training accuracy would be much higher than the test accuracy, but here both values are almost indentical, if the model was underfitting, both accuracies would be low, but here the accuracy is reasonably good around 72%.

So overall, the model is well balanced and is performing consistently on both training and testing data, which means it has good generalization ability.

# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

### **Modified Model**

In [ ]:
class DiabetesModelV2(nn.Module):
  def __init__(self, num_features):
    super(DiabetesModelV2, self).__init__()

    self.linear1 = nn.Linear(num_features, 32)  # hidden layer
    self.relu = nn.ReLU()

    self.linear2 = nn.Linear(32, 1)                      # output layer
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear1(features)
    out = self.relu(out)
    out = self.linear2(out)
    out = self.sigmoid(out)
    return out

In [ ]:
modelV2 = DiabetesModelV2(num_features=8)

loss_fn2 = nn.BCELoss()

# Optimizer
optimizer2 = optim.Adam(modelV2.parameters(), lr=0.001)

In the previous model, I used a hidden layer with 16 neurons. In this new version, I increased the numer of neurons to 32 in the hidden layer to give the model more learning capacity. I kept all other settings the same, such as:
* Same dataset
* Same number of epochs (100)
* Same optimizerr (Adam with learning rate 0.001)
* Same loss function (BCELoss)

In [ ]:
epochs = 100
loss_history_v2 = []

for epoch in range(epochs):
  # Gradient reset
  optimizer2.zero_grad()

  # Forward pass
  outputs = modelV2(X_train)

  # Loss computation
  loss = loss_fn2(outputs, y_train.view(-1, 1))

  # Backward pass
  loss.backward()

  # Update parameters
  optimizer2.step()

  # Store loss
  loss_history_v2.append(loss.item())

  # Print loss every 10 epochs
  if (epoch+1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item(): .4f}")

Epoch [10/100], Loss:  0.6636
Epoch [20/100], Loss:  0.6389
Epoch [30/100], Loss:  0.6165
Epoch [40/100], Loss:  0.5962
Epoch [50/100], Loss:  0.5778
Epoch [60/100], Loss:  0.5612
Epoch [70/100], Loss:  0.5465
Epoch [80/100], Loss:  0.5335
Epoch [90/100], Loss:  0.5220
Epoch [100/100], Loss:  0.5119


### **Training Observation (Convergence Speed)**

During training, I stored the loss values for each epoch in `loss_history_v2`. By observing the loss values every 10 epochs, I noticed that the loss decreased smoothly and slightly faster in the early epochs compared to the previous model.

This shows that the model with 32 hidden neurons was able to learn patterns a bit more efficiently at the beginning of training. This convergence behavior was stable, and there were no sudden fluctuations in loss.

In [ ]:
modelV2.eval()

with torch.no_grad():
    outputs = modelV2(X_test)

    predicted = (outputs >= 0.5).float()

    corr = (predicted == y_test.view(-1, 1)).sum().item()
    total = y_test.size(0)

    new_accuracy = corr / total
    print("New Test Accuracy:", new_accuracy)

New Test Accuracy: 0.7402597402597403


In [ ]:
print(f"Previous Accuracy:  {accuracy}")
print("New Accuracy:", new_accuracy)

if new_accuracy >  accuracy:
    print("Model performance improved")
else:
    print("No improvement or performance decreased")

Previous Accuracy:  0.7272727272727273
New Accuracy: 0.7402597402597403
Model performance improved


### **Final Performance Comparison**

After training, I evaluated both models on the test dataset:
* Previous Model Accuracy: 0.7272
* New Model Accuracy: 0.7402

So, the modified model shows a clear improvement in final performance

### **Explanaiton of the Change**

After increasing the number of hidden neurons, the model showed slightly better learning ability. The loss decreased smoothly during training, and the final test accuracy also improved.This happend because the model now has more neurons, so it can capture more patterns and relationships in the dataset.

However, the improvement is not very large because the dataset is relatively small and simple, so the previous model was already performing reasonably well. Still, the modified model provides better generalization and slightly improved performance without causing overfitting.

# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:

**Why must gradients be reset every epoch?**

In my training loop, I used `optimizer.zero_grad()` in every epoch. This is important because PyTorch does not automatically remove old gradients. It keeps adding new gradients on top of the previous ones. If I do not reset them, the model will use incorrect accumulated values, and the weight updates will become wrong. This can make the training unstable and reduce the model's performance. So, resetting gradients ensures that every epoch learns only from the current batch of computations.

---

**What happens if learning rate is too high?**

If the learning rate is too high, the model will learn too quickly and may skip the best solution. Instead of slowly reaching the minimum loss, it will keep jumping around it. This can make the training unstable, and the loss may even increase instead of decreasing. In the worst case, the model may fail to converge at all. In my experiment, I used a small learning rate (0.001 with Adam), which helped the model learn steadily and improve accuracy.

---

**What happens if learning rate is too small?**

If the learning rate is too small, the model learns very slowly, The loss decrease very gradually, and it may take many more epochs to reach a good result. Even through training becomes stable, it is inefficient and time-consuming. The model might also get stuck in a situation where impovement is very small. So, a balanced learning rate like 0.001 works well for this dataset because it gives stable and steadly learning.

---

**Why do we define layers inside the constructor (init) and not inside forward()?**

In my model, I defined all layers inside the `__init__()` function because this is where PyTorch registers the model parameters. When layers are defined in the constructor, PyTorch can track them and update their weights during training. If I define layers inside the `forward()` function, they would be created again and again for every input, and the model would not actually learn anything beacuse parameters would not be properly stored or updated. So, defining layers in `__init__()` is necessary for correct training.

